<div style="display:flex; align-items:center; gap:18px; text-align:left">
<img src="https://sebastiancontz.github.io/ust-introduccion-machine-learning/assets/logo-ust.svg" width="100" alt="Logo de la Universidad Santo Tomás">
<div>
<p>Ingeniería en Información y Control de Gestión</p>
<p>Facultad de Economía y Negocios</p>
<p>Introducción a Machine Learning</p>
<p>Repaso de la Unidad 1: Fundamentos de Machine Learning</p>
</div>
</div>

# Repaso · Actividades de la Unidad 1

Dos actividades para practicar con datos los conceptos de la Unidad 1. Cada paso se responde
escribiendo o completando los espacios marcados `?????`.

Las dos trabajan sobre `absentismo_laboral.csv`, que **no** es el archivo que verán en la
evaluación. Es a propósito: lo que hay que llevarse es el procedimiento, no las cifras de un
archivo.

**Cómo trabajar.** Hagan una copia de este notebook en su propia unidad. Resuelvan primero sin
consultar. Los **criterios de éxito** y las **respuestas comentadas** de las dos actividades están
en la [guía del repaso](https://sebastiancontz.github.io/ust-introduccion-machine-learning/ediciones/2026/guias/repaso-solemne-01.html), plegadas: ábranlas después de intentarlo.

Cada decisión va con su justificación escrita. Una columna bien descartada sin el criterio que la
descarta no muestra el razonamiento que la unidad pide.

## Preparación del entorno

En Colab estas librerías ya vienen instaladas, así que la celda siguiente termina en segundos.
Está igual porque el notebook también tiene que correr fuera de Colab.

In [ ]:
%%capture
!pip install -q pandas numpy scikit-learn

---

## Actividad 1 · Formular y fijar el piso

**Nivel básico. Cuatro pasos.**

Una empresa de mensajería tiene un problema de cobertura de turnos: cuando alguien falta, el
despacho se reorganiza a última hora y ese día se atrasan las entregas. La jefatura de operaciones
quiere anticipar **cuántas horas de ausencia** va a tener cada episodio para reorganizar el turno
antes de que el día empiece, en vez de a mitad de mañana. Les piden formular el problema antes de
que nadie entrene un modelo.

El archivo `absentismo_laboral.csv` tiene registros de ausencias del personal, con las variables del
empleado y del período de cada ausencia. La celda siguiente lo carga en la variable `datos`.

**Preparación.** Ejecuten esta celda antes de responder: carga el archivo y muestra su estructura.

In [ ]:
import numpy as np
import pandas as pd

RUTA = (
    "https://raw.githubusercontent.com/sebastiancontz/ust-introduccion-machine-learning-colab/main/ediciones/2026/datasets/absentismo_laboral.csv"
)
datos = pd.read_csv(RUTA)
datos.info()

### Paso 1 · La ficha de formulación

Respondan las tres preguntas, una por línea. Cada una se sostiene con lo que el archivo muestra, no
con lo que su nombre sugiere.

1. ¿Qué representa un registro de este archivo? Sosténganlo con dos conteos.
2. ¿Cuál es la variable objetivo y en qué unidad está medida?
3. ¿En qué momento tiene que estar lista la predicción para que la jefatura alcance a reorganizar
   el turno?

### Respuesta escrita — paso 1

_Escriban su respuesta acá._

### Paso 2 · El piso

El baseline es la referencia contra la que se compara todo modelo. Acá se usa la regla más simple:
predecir siempre el mismo valor para todos los episodios.

1. Completen la celda. Usen la **media** como valor constante, para que todo el curso reporte la
   misma referencia.
2. Reporten el error del baseline como el **promedio del error absoluto, en horas por episodio**.
3. Expliquen en una o dos frases qué le dice esa cifra a quien tiene que reorganizar el turno, y
   qué **no** le dice sobre qué episodio cubrir primero.

In [ ]:
# complete los espacios marcados abajo
objetivo = datos["?????"]

# el baseline predice siempre el mismo valor para todos los episodios
piso = objetivo.?????()

error_promedio = np.mean(np.abs(objetivo - piso))
print(f"baseline: {piso:.2f} horas · error promedio: {error_promedio:.2f} horas")

### Respuesta escrita — paso 2

_Escriban su respuesta acá._

### Paso 3 · Un registro que no es lo que parece

La columna `falta_disciplinaria` marca un subconjunto de registros. Antes de seguir usando el piso
del paso 2, hay que saber qué son esos registros.

1. Completen la celda: cuenten cuántos son y miren con qué horas y con qué motivo aparecen.
2. Completen la segunda parte, que recalcula el mismo piso dejando fuera esos registros, y reporten
   las dos cifras nuevas.
3. Expliquen qué son esos registros, si pertenecen o no a la unidad de observación que declararon
   en el paso 1, y **qué cambió realmente** entre las dos cifras del piso. Cuidado con la lectura
   fácil: la pregunta no es si el error bajó.

In [ ]:
# complete los espacios marcados abajo

# (a) que son los registros marcados
disciplinarios = datos["falta_disciplinaria"].eq("si")
print(f"registros marcados: {disciplinarios.sum()}")
print("horas:", sorted(datos.loc[disciplinarios, "?????"].unique().tolist()))
print("motivos:", datos.loc[disciplinarios, "motivo_ausencia"].unique().tolist())

# (b) el mismo piso, ahora sin ellos
episodios = datos.loc[~disciplinarios]
piso_episodios = episodios["horas_ausencia"].?????()
error_episodios = np.mean(np.abs(episodios["horas_ausencia"] - piso_episodios))
print(f"{len(episodios)} registros · baseline: {piso_episodios:.2f} horas "
      f"· error promedio: {error_episodios:.2f} horas")

### Respuesta escrita — paso 3

_Escriban su respuesta acá._

### Paso 4 · Dos columnas que quedan fuera, por dos motivos distintos

La jefatura propone dos columnas para el modelo:

1. `motivo_ausencia`, que registra por qué faltó la persona.
2. `indice_masa_corporal`, que el sistema de salud ocupacional ya tiene registrado para cada
   empleado desde antes.

Para cada una, decidan si entra a la ficha de formulación y **nombren el criterio** que la resuelve.
Los dos criterios **no** son el mismo. Digan además qué otras columnas del archivo caen bajo el
mismo criterio que la segunda.

### Respuesta escrita — paso 4

_Escriban su respuesta acá._

---

## Actividad 2 · Cada columna entra representada, o no entra

**Nivel avanzado. Cuatro pasos.**

Una empresa de mensajería quiere anticipar cuántas horas de ausencia va a tener cada episodio, para
reorganizar el turno antes de que el día empiece. La revisión de calidad del archivo ya se hizo. Lo
que falta es la decisión que les encargan a ustedes: **cómo entra cada columna al preprocesador**, y
qué pasa con las que no entran.

Acá **no se entrena ningún modelo y no se reporta ninguna cifra de error**: lo que se practica es la
decisión de representación y cómo se sostiene con el archivo a la vista.

La celda de preparación deja los datos en `datos`, declara el objetivo en `OBJETIVO` y entrega el
diccionario `ORDEN_EDUCACION`, que es del negocio y no se descubre en los datos.

**Preparación.** Ejecuten esta celda antes de responder.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

RUTA = (
    "https://raw.githubusercontent.com/sebastiancontz/ust-introduccion-machine-learning-colab/main/ediciones/2026/datasets/absentismo_laboral.csv"
)
datos = pd.read_csv(RUTA)

# el orden de nivel_educacion viene dado: es del negocio, no se descubre en los datos
ORDEN_EDUCACION = {
    "media": 0,
    "universitaria": 1,
    "posgrado": 2,
    "magister o doctorado": 3,
}

# la variable objetivo, que el enunciado ya declara
OBJETIVO = "horas_ausencia"

print(f"{len(datos)} registros y {datos.shape[1]} columnas")

### Paso 1 · Repartir las columnas en las tres rutas

El preprocesador ya viene armado con sus tres rutas: una codifica, una escala y una deja pasar la
columna ya numerada. Lo que falta es la decisión de representación: qué columna recibe qué
tratamiento. Ninguna de las tres listas está completa.

1. Completen `NOMINALES` con las **dos** categóricas sin orden que describen **cuándo** ocurrió la
   ausencia.
2. Completen `ORDINAL` con la única columna de texto cuyo orden es **real**, que es la que
   `ORDEN_EDUCACION` sabe numerar.
3. Completen `NUMERICAS` con **cuatro** columnas numéricas del empleado: su edad, su antigüedad,
   cuántos hijos tiene y a qué distancia vive del trabajo.
4. Ejecuten la celda y reporten las **dos** cifras que imprime: cuántas columnas entran al
   preprocesador y cuántas salen. Digan además cuántas columnas aporta cada ruta a la salida, y por
   qué solo una de las tres ensancha la matriz.

El preprocesador recibe **todas** las predictoras candidatas: la celda le entrega el archivo sin la
variable objetivo y sin la versión de texto de la ordinal, que ya quedó numerada. La ruta `"ord"`
usa `"passthrough"` porque esa columna entra sin otra transformación.

In [ ]:
# complete los espacios marcados abajo
NOMINALES = ["?????", "?????"]
ORDINAL = "?????"
NUMERICAS = ["?????", "?????", "?????", "?????"]

datos["educacion_num"] = datos[ORDINAL].map(ORDEN_EDUCACION)

preprocesador = ColumnTransformer([
    ("nom", OneHotEncoder(handle_unknown="ignore"), NOMINALES),
    ("num", StandardScaler(), NUMERICAS),
    ("ord", "passthrough", ["educacion_num"]),
])

# el preprocesador recibe todas las predictoras candidatas, no solo las nombradas arriba
X = datos.drop(columns=[OBJETIVO, ORDINAL])
matriz = preprocesador.fit_transform(X)

print(f"entran {X.shape[1]} columnas y salen {matriz.shape[1]}")
for columna in NOMINALES:
    print(f"  {columna}: {datos[columna].nunique()} categorías")

### Respuesta escrita — paso 1

_Escriban su respuesta acá._

### Paso 2 · Lo que el preprocesador descartó sin aviso

Entraron más columnas de las que salieron. El valor por defecto de `remainder` es `"drop"`: toda
columna que llega al preprocesador y no aparece en ninguna de las rutas **se descarta, sin aviso y
sin error**.

1. Completen la celda con la columna nombrada que falta en la lista, y reporten cuántas columnas se
   descartaron.
2. Comprueben que el desglose por ruta que imprime la celda coincide con la cifra de salida del
   paso 1.
3. Separen la lista de descartadas en **dos grupos, con al menos tres columnas en cada uno**: las
   que corresponde dejar fuera, y las que se **perdieron por omisión** aunque estaban disponibles
   en el momento de decidir y eran utilizables.
4. Escriban el motivo de cada columna del primer grupo. Los motivos **no son todos el mismo**: hay
   un identificador, hay una columna temporal, hay una que se conoce después del hecho, hay una que
   vale exactamente cuando el objetivo vale cero, hay variables que no son legítimas para fundar
   una decisión laboral y hay cifras que no describen al empleado. Si alguno de esos motivos les
   parece discutible, díganlo y sostengan de qué depende.

In [ ]:
# complete el espacio marcado abajo
nombradas = NOMINALES + NUMERICAS + ["?????"]

sin_nombrar = [columna for columna in X.columns if columna not in nombradas]

print(f"{len(sin_nombrar)} columnas entraron y el preprocesador las descartó sin aviso:")
for columna in sin_nombrar:
    print(" ", columna)

# cuantas columnas aporta cada ruta a la salida
del_one_hot = sum(datos[columna].nunique() for columna in NOMINALES)
print(f"\none-hot: {del_one_hot} · escaladas: {len(NUMERICAS)} · ordinal: 1 "
      f"· total: {del_one_hot + len(NUMERICAS) + 1}")

### Respuesta escrita — paso 2

_Escriban su respuesta acá._

### Paso 3 · Una columna numérica que en realidad es categórica

`mes` es el mes calendario de la ausencia. Está disponible desde antes de que el mes empiece, así
que es una predictora legítima, y ninguna de las tres rutas del paso 1 la nombró.

1. Numerar una categórica afirma **dos** cosas distintas sobre ella. Digan cuáles son las dos, y
   cuál de las dos se puede verificar con este archivo y cuál no.
2. Ejecuten la celda: hay un dato de `mes` que decide el tratamiento y que no se ve en el
   diccionario de datos. Digan qué es y qué hay que hacer con él.
3. Decidan un tratamiento para `mes` —entra como numérica tal como viene, o entra con one-hot— y
   sostengan la decisión.
4. Digan con cuántas columnas queda la salida del paso 1 bajo cada uno de los dos tratamientos.

In [ ]:
print("valores distintos de mes:", sorted(datos["mes"].unique().tolist()))
print("registros por valor:")
print(datos["mes"].value_counts().sort_index().to_string())

### Respuesta escrita — paso 3

_Escriban su respuesta acá._

### Paso 4 · Alta cardinalidad, identificadores y un motivo que no es ninguno de los dos

`motivo_ausencia` e `id_empleado` quedaron fuera del preprocesador, y no por el mismo motivo.
Ejecuten la celda y respondan.

1. Digan con cuántas columnas quedaría la salida del paso 1 si `motivo_ausencia` entrara con
   one-hot, y por qué ese ancho es un problema para un modelo que estima un coeficiente por
   columna. Miren también con cuántos registros aparece cada categoría.
2. Nombren la salida que la unidad propone para aliviar la alta cardinalidad, y digan qué se pierde
   al aplicarla.
3. Digan por qué esa salida **no** resuelve el caso de `motivo_ausencia`, y por qué tampoco
   resuelve el de `id_empleado`. Los dos quedan fuera, y la cardinalidad no es el motivo de
   ninguno de los dos.

In [ ]:
for columna in ("motivo_ausencia", "id_empleado"):
    print(f"{columna}: {datos[columna].nunique()} categorías distintas")

print("\ncategorías de motivo_ausencia con 2 registros o menos:")
frecuencias = datos["motivo_ausencia"].value_counts()
print(frecuencias[frecuencias <= 2].to_string())

### Respuesta escrita — paso 4

_Escriban su respuesta acá._

---

## Antes de cerrar

Vuelvan a la [guía del repaso](https://sebastiancontz.github.io/ust-introduccion-machine-learning/ediciones/2026/guias/repaso-solemne-01.html) y abran, ahora sí, los criterios de éxito y las respuestas
comentadas de las dos actividades. Comparen **el razonamiento**, no solo la cifra: en las dos hay un
paso donde la respuesta correcta no es la lectura más rápida del resultado.